# get_logits_cos_sims — Attention Alignment Variant

Sequence-aware version of `get_logits_cos_sims.ipynb`.

**What changed:** the original notebook trains a plain MLP ("alignment network") on
*flattened, zero-padded* PanPhon feature vectors `[loan | orig]`. This variant keeps the
per-position PanPhon sequences and trains an **attention pair model**: each position of the
loan word attends to positions of the original word (and vice versa), so the network learns
an explicit phoneme alignment between the two words. Training uses mini-batches and early
stopping on the validation split.

**What did NOT change:** the interface. The model still emits one logit per word pair, saved
as the `DNN_logits` column of the production CSVs, so `evaluate-experiments.ipynb` works
unmodified. MBERT/XLM cosine similarities are computed exactly as before.

Assumes you have run `make-train-test-splits.ipynb` first to make the `alldata`, `realdist`,
and `balanced` train/test splits for the chosen language pair.


# Imports and setup

In [ ]:
#if you are running this with collab un comment the installations
!pip install numpy pandas tqdm seaborn matplotlib scikit-learn requests panphon editdistance epitran eng-to-ipa googletrans safetensors transformers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import panphon
import panphon.distance
import editdistance # levenshtein
import epitran
import eng_to_ipa as eng
from epitran.backoff import Backoff
from googletrans import Translator
from sklearn.utils import shuffle
from sklearn.linear_model import LogisticRegression
epitran.download.cedict()

In [ ]:
import torch
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from matplotlib import rc
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import os
import torch
from torch import nn
from torch import nn, optim

import torch.nn.functional as F

%matplotlib inline

In [ ]:
import sklearn as sk
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np
import io
import requests
import csv

In [ ]:
from transformers import (
    BertTokenizer,
    BertModel,
    get_linear_schedule_with_warmup,
    BertForSequenceClassification,
    BertForPreTraining,
    AutoModel
)

from torch.optim import AdamW

In [ ]:
# NOTE: the old XLM-100 / MBERT imports were removed.
# The two semantic-similarity models are now loaded via AutoTokenizer/AutoModel
# (see the "Download LMs" cell): jhu-clsp/mmBERT-base and Alibaba-NLP/gte-multilingual-base.
# They require transformers >= 5.20 (modernbert + gte model types) and torch >= 2.6.

In [ ]:
torch.cuda.empty_cache()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

#Additional Info when using cuda
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print('Memory Usage:')
    print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

#device = torch.device("cuda:0:3" if torch.cuda.is_available() else "cpu") ## specify the GPU id's, GPU id's start from 0.

In [ ]:
if torch.cuda.is_available():
    print(torch.cuda.get_device_name())

# Attention Pair Model (sequence-aware)


In [ ]:
class AttentionPairModel(nn.Module):
    """Sequence-aware loanword classifier with learned phoneme alignment.

    Input: per-position PanPhon feature sequences of the loan word and the original
    word, plus padding masks (1 = valid position).

    Architecture:
        loan (B,T1,F) --Linear--> (B,T1,H)
        orig (B,T2,F) --Linear--> (B,T2,H)
        loan positions cross-attend to orig positions (and vice versa)
        masked-mean pooling of raw + aligned reps -> concat (4H) -> MLP -> logit

    The cross-attention weights are the learned alignment between the two words;
    they are stored in `self.last_alignment` for inspection/visualization.

    forward returns (sigmoid_prob, raw_logits, alignment) — index [1] is the raw
    logit, same convention as the original NeuralNetwork.
    """

    def __init__(self, feat_dim=24, hidden=64, heads=4, mlp_hidden=128, dropout=0.3):
        super(AttentionPairModel, self).__init__()
        self.loan_emb = nn.Linear(feat_dim, hidden)
        self.orig_emb = nn.Linear(feat_dim, hidden)
        self.attn_loan2orig = nn.MultiheadAttention(hidden, heads, batch_first=True, dropout=dropout)
        self.attn_orig2loan = nn.MultiheadAttention(hidden, heads, batch_first=True, dropout=dropout)
        self.classifier = nn.Sequential(
            nn.Linear(4 * hidden, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1),
        )
        self.last_alignment = None

    @staticmethod
    def _masked_mean(x, mask):
        # x: (B,T,H), mask: (B,T) with 1 for valid positions
        m = mask.unsqueeze(-1).float()
        return (x * m).sum(1) / m.sum(1).clamp(min=1.0)

    def forward(self, loan, orig, loan_mask, orig_mask):
        l = self.loan_emb(loan)
        o = self.orig_emb(orig)
        l_kpm = ~loan_mask.bool()
        o_kpm = ~orig_mask.bool()
        l_att, w_l2o = self.attn_loan2orig(l, o, o, key_padding_mask=o_kpm,
                                            need_weights=True, average_attn_weights=True)
        o_att, w_o2l = self.attn_orig2loan(o, l, l, key_padding_mask=l_kpm,
                                           need_weights=True, average_attn_weights=True)
        self.last_alignment = (w_l2o, w_o2l)  # (B,T1,T2) and (B,T2,T1)
        x = torch.cat([self._masked_mean(l, loan_mask),
                       self._masked_mean(o, orig_mask),
                       self._masked_mean(l_att, loan_mask),
                       self._masked_mean(o_att, orig_mask)], dim=1)
        logits = self.classifier(x).squeeze(-1)
        return torch.sigmoid(logits), logits, self.last_alignment

    def fit(self, loan_tr, orig_tr, lm_tr, om_tr, y_tr,
            loan_val, orig_val, lm_val, om_val, y_val,
            n_epochs=100, batch_size=256, lr=0.001, weight_decay=1e-4,
            early_stop_patience=15, verbose_every=5):
        """Mini-batch training with early stopping on validation loss."""
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(self.parameters(), lr=lr, weight_decay=weight_decay)
        n = len(y_tr)
        best_val, best_state, bad_epochs = float('inf'), None, 0
        history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

        for epoch in range(n_epochs):
            self.train()
            perm = torch.randperm(n, device=loan_tr.device)
            epoch_loss, correct = 0.0, 0
            for i in range(0, n, batch_size):
                idx = perm[i:i + batch_size]
                logits = self(loan_tr[idx], orig_tr[idx], lm_tr[idx], om_tr[idx])[1]
                loss = criterion(logits, y_tr[idx])
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item() * len(idx)
                correct += int(((logits >= 0) == (y_tr[idx] >= 0.5)).sum())

            self.eval()
            with torch.no_grad():
                vlogits = self(loan_val, orig_val, lm_val, om_val)[1]
                vloss = criterion(vlogits, y_val).item()
                vacc = float(((vlogits >= 0) == (y_val >= 0.5)).float().mean())

            tloss = epoch_loss / n
            tacc = correct / n
            history['train_loss'].append(tloss)
            history['val_loss'].append(vloss)
            history['train_acc'].append(tacc)
            history['val_acc'].append(vacc)

            if epoch % verbose_every == 0 or epoch == n_epochs - 1:
                print(f"epoch {epoch:3d} | train loss {tloss:.4f} acc {tacc:.3f} | "
                      f"val loss {vloss:.4f} acc {vacc:.3f}")

            if vloss < best_val - 1e-4:
                best_val = vloss
                best_state = {k: v.detach().clone() for k, v in self.state_dict().items()}
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= early_stop_patience:
                    print(f"early stopping at epoch {epoch} (best val loss {best_val:.4f})")
                    break

        if best_state is not None:
            self.load_state_dict(best_state)
        self.eval()
        return history

    @staticmethod
    def plot_history(history):
        epochs = range(1, len(history['train_loss']) + 1)
        plt.plot(epochs, history['train_loss'], label='Train loss')
        plt.plot(epochs, history['val_loss'], label='Val loss')
        plt.title('Training and validation loss')
        plt.legend()
        plt.show()
        plt.figure()
        plt.plot(epochs, history['train_acc'], label='Train acc')
        plt.plot(epochs, history['val_acc'], label='Val acc')
        plt.title('Training and validation accuracy')
        plt.legend()
        plt.show()


# MyDataset definition

In [ ]:
# Overriding the Dataset class required for the use of PyTorch's data loader classes.
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, l1_encodings, l2_encodings):
        self.l1_encodings = l1_encodings
        self.l2_encodings = l2_encodings

    def __getitem__(self, idx):
        item = {('l1_' + key): torch.tensor(val[idx]) for key, val in self.l1_encodings.items()}
        item2 = {('l2_' + key): torch.tensor(val[idx]) for key, val in self.l2_encodings.items()}
        item.update(item2)
        # item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.l1_encodings['attention_mask'])

In [ ]:
!pip install -U sacremoses

# Download LMs (mmBERT + GTE)

In [ ]:
# Modern multilingual embedding models (replacing MBERT and XLM-100).
# Downloaded from the Hugging Face Hub on first use.
PRE_TRAINED_bert_MODEL = 'jhu-clsp/mmBERT-base'              # ModernBERT architecture
PRE_TRAINED_xlm_MODEL  = 'Alibaba-NLP/gte-multilingual-base' # GTE architecture, XLM-R-style tokenizer

# Preload so the similarity cells below don't stall on downloads.
mmbert_tokenizer = AutoTokenizer.from_pretrained(PRE_TRAINED_bert_MODEL)
mmbert_model = AutoModel.from_pretrained(PRE_TRAINED_bert_MODEL)
gte_tokenizer = AutoTokenizer.from_pretrained(PRE_TRAINED_xlm_MODEL)
gte_model = AutoModel.from_pretrained(PRE_TRAINED_xlm_MODEL)

In [ ]:
xlm_model

# Pipeline function definitions

## Get PanPhon phonetic sequences


In [ ]:
# PanPhon feature table (shared) — per-position feature vectors, NOT flattened
FT = panphon.FeatureTable()
FEAT_DIM = len(FT.word_to_vector_list('a', numeric=True)[0])
print(f"PanPhon feature dim: {FEAT_DIM}")

def get_panphon_sequences(train_set, test_set):
    # Get per-position PanPhon feature sequences for loan and original words.
    # (Empty transcriptions get a single zero position so masks always have
    # at least one valid position — keeps attention numerically safe.)
    def seqs(w):
        v = FT.word_to_vector_list(w, numeric=True)
        return v if v else [[0.0] * FEAT_DIM]
    train_set['seq_loan'] = train_set['loan_word_epitran'].apply(seqs)
    train_set['seq_orig'] = train_set['original_word_epitran'].apply(seqs)
    test_set['seq_loan'] = test_set['loan_word_epitran'].apply(seqs)
    test_set['seq_orig'] = test_set['original_word_epitran'].apply(seqs)


In [ ]:
def pad_panphon_sequences(train_set, test_set, maxlen):
    # Pad sequences to (maxlen, FEAT_DIM) and build padding masks (1 = valid).
    # maxlen is a tuple (loan_maxlen, orig_maxlen), as in the original notebook.
    # Idempotent: already-padded columns are left alone (safe to re-run the cell).
    def pad_one(seq, T_max):
        T = max(1, min(len(seq), T_max))
        mat = np.zeros((T_max, FEAT_DIM), dtype=np.float32)
        mat[:T] = np.array(seq[:T], dtype=np.float32)
        mask = np.zeros(T_max, dtype=np.float32)
        mask[:T] = 1.0
        return [mat, mask]
    def needs_padding(col):
        s = col.iloc[0]
        return not (len(s) == 2 and isinstance(s[0], np.ndarray) and s[0].ndim == 2)
    for df in (train_set, test_set):
        if needs_padding(df['seq_loan']):
            df['seq_loan'] = df['seq_loan'].apply(lambda s: pad_one(s, maxlen[0]))
        if needs_padding(df['seq_orig']):
            df['seq_orig'] = df['seq_orig'].apply(lambda s: pad_one(s, maxlen[1]))


## Add target labels and make train and validation sets

In [ ]:
def add_target_labels(train_set, test_set):
    Y_train = np.array([y for y in train_set['label_bin']])
    Y_test = np.array([y for y in test_set['label_bin']])
    return Y_train, Y_test

Make an index-based validation split for training the attention model


In [ ]:
def make_train_val_idx(train_set, Y_train):
    # Index-based 80/20 stratified split so loan/orig sequences and masks stay
    # in sync (replaces the flat-array hstack split of the original notebook).
    idx = np.arange(len(train_set))
    tr, va = train_test_split(idx, test_size=0.2, random_state=1, stratify=Y_train)
    return tr, va


Make tensors

In [ ]:
def seq_tensors(df, idx=None):
    # Build (loan, orig, loan_mask, orig_mask) tensors of shape (N, T, FEAT_DIM)
    if idx is not None:
        df = df.iloc[idx]
    loan = torch.tensor(np.stack([s[0] for s in df['seq_loan']]), dtype=torch.float32).to(device)
    orig = torch.tensor(np.stack([s[0] for s in df['seq_orig']]), dtype=torch.float32).to(device)
    lm = torch.tensor(np.stack([s[1] for s in df['seq_loan']]), dtype=torch.float32).to(device)
    om = torch.tensor(np.stack([s[1] for s in df['seq_orig']]), dtype=torch.float32).to(device)
    return loan, orig, lm, om

def get_logits(model, df):
    # One raw logit per row — same output convention as the original DNN logits.
    model.eval()
    with torch.no_grad():
        logits = model(*seq_tensors(df))[1]
    return logits.detach().cpu().numpy()


## Get cosine similarities

## mmBERT — `jhu-clsp/mmBERT-base` (stored as `MBERT_cos_sim`)

In [ ]:
def get_mbert_cos_sims(l1_data, l2_data):
    """Masked mean-pooled cosine similarity with jhu-clsp/mmBERT-base (ModernBERT).
    NOTE: [CLS] pooling is degenerate for this MLM checkpoint (all sims ~0.999),
    so we use attention-masked mean pooling instead."""
    with torch.no_grad():
        tokenizer = AutoTokenizer.from_pretrained(PRE_TRAINED_bert_MODEL)
        tokenizer.model_max_length = MAXTOKENS
        l1_encodings = tokenizer(list(l1_data), truncation=True, padding=True, max_length=MAXTOKENS, return_tensors="pt")
        l2_encodings = tokenizer(list(l2_data), truncation=True, padding=True, max_length=MAXTOKENS, return_tensors="pt")

        dataset = MyDataset(l1_encodings, l2_encodings)

        data_loader = DataLoader(dataset, batch_size=BS, shuffle=False)  # shuffle False for reproducibility

        base_model = AutoModel.from_pretrained(PRE_TRAINED_bert_MODEL).to(device)
        base_model.eval()
        cos_s = torch.nn.CosineSimilarity()

        def pool(h, m):
            h = h * m.unsqueeze(-1).float()
            return torch.nn.functional.normalize(h.sum(1) / m.sum(1).unsqueeze(-1).float(), dim=1)

        sim_lst = []

        #loop through dataset
        for step, batch in enumerate(data_loader):
            l1_h = base_model(batch['l1_input_ids'].to(device),
                              attention_mask=batch['l1_attention_mask'].to(device),
                              return_dict=True).last_hidden_state
            l2_h = base_model(batch['l2_input_ids'].to(device),
                              attention_mask=batch['l2_attention_mask'].to(device),
                              return_dict=True).last_hidden_state
            l1_vector = pool(l1_h, batch['l1_attention_mask'].to(device))
            l2_vector = pool(l2_h, batch['l2_attention_mask'].to(device))
            sims = cos_s(l1_vector, l2_vector).data.cpu().numpy()
            sim_lst.extend(list(sims))
            if (step * BS) % 100 < BS:
                print("Got {}".format(len(sim_lst)))
        print()

    return sim_lst

## GTE — `Alibaba-NLP/gte-multilingual-base` (stored as `XLM_cos_sim`)

In [ ]:
def get_xlm_cos_sims(l1_data, l2_data):
    """[CLS] cosine similarity with Alibaba-NLP/gte-multilingual-base (GTE)."""
    with torch.no_grad():
        tokenizer = AutoTokenizer.from_pretrained(PRE_TRAINED_xlm_MODEL)
        tokenizer.model_max_length = MAXTOKENS
        l1_encodings = tokenizer(list(l1_data), truncation=True, padding=True, max_length=MAXTOKENS, return_tensors="pt")
        l2_encodings = tokenizer(list(l2_data), truncation=True, padding=True, max_length=MAXTOKENS, return_tensors="pt")

        dataset = MyDataset(l1_encodings, l2_encodings)

        data_loader = DataLoader(dataset, batch_size=BS, shuffle=False)  # shuffle False for reproducibility

        base_model = AutoModel.from_pretrained(PRE_TRAINED_xlm_MODEL).to(device)
        base_model.eval()
        cos_s = torch.nn.CosineSimilarity()

        sim_lst = []

        #loop through dataset
        for step, batch in enumerate(data_loader):
            l1_vector = base_model(batch['l1_input_ids'].to(device),
                                          attention_mask=batch['l1_attention_mask'].to(device),
                                          return_dict=True).last_hidden_state[:, 0, :]
            l2_vector = base_model(batch['l2_input_ids'].to(device),
                                          attention_mask=batch['l2_attention_mask'].to(device),
                                          return_dict=True).last_hidden_state[:, 0, :]
            sims = cos_s(l1_vector, l2_vector).data.cpu().numpy()
            sim_lst.extend(list(sims))
            if (step * BS) % 100 < BS:
                print("Got {}".format(len(sim_lst)))
        print()

    return sim_lst

# Load `language-pairs.json` list and run pipeline for each

In [ ]:
pairs = None

with open('../language-pairs.json', 'r') as f:
    pairs = json.loads(f.read())

for pair in pairs:
    print(pair)
    L1 = pairs[pair]['target']['name']
    L2 = pairs[pair]['source']['name']

    # load datasets
    prefix = f'../Datasets/train_test_split/{L1}-{L2}'

    train_alldata = pd.read_csv(f'{prefix}/alldata/{L1}-{L2}-train_production_alldata.csv')
    test_alldata = pd.read_csv(f'{prefix}/alldata/{L1}-{L2}-test_production_alldata.csv')

    train_realdist = pd.read_csv(f'{prefix}/realdist/{L1}-{L2}-train_production_realdist.csv')
    test_realdist = pd.read_csv(f'{prefix}/realdist/{L1}-{L2}-test_production_realdist.csv')

    train_balanced = pd.read_csv(f'{prefix}/balanced/{L1}-{L2}-train_production_balanced.csv')
    test_balanced = pd.read_csv(f'{prefix}/balanced/{L1}-{L2}-test_production_balanced.csv')

    # get PanPhon sequences for alldata split and pad to pair-specific max lengths
    get_panphon_sequences(train_alldata, test_alldata)
    alldata_maxlen = (max(max(len(s) for s in train_alldata['seq_loan']),
                          max(len(s) for s in test_alldata['seq_loan'])),
                      max(max(len(s) for s in train_alldata['seq_orig']),
                          max(len(s) for s in test_alldata['seq_orig'])))
    pad_panphon_sequences(train_alldata, test_alldata, alldata_maxlen)

    # add target labels
    Y_train, Y_test = add_target_labels(train_alldata, test_alldata)

    # make train and val index splits
    idx_tr, idx_va = make_train_val_idx(train_alldata, Y_train)
    loan_tr, orig_tr, lm_tr, om_tr = seq_tensors(train_alldata, idx_tr)
    loan_va, orig_va, lm_va, om_va = seq_tensors(train_alldata, idx_va)
    y_tr = torch.tensor(train_alldata['label_bin'].values[idx_tr], dtype=torch.float32).to(device)
    y_va = torch.tensor(train_alldata['label_bin'].values[idx_va], dtype=torch.float32).to(device)

    # instantiate model
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nUsing {device} device\n")

    # set random seeds for reproducibility
    torch.manual_seed(666)
    np.random.seed(666)

    model = AttentionPairModel(feat_dim=FEAT_DIM, hidden=64, heads=4, mlp_hidden=128, dropout=0.3).to(device)
    print(model, "\n")

    # train with early stopping on the validation split
    history = model.fit(loan_tr, orig_tr, lm_tr, om_tr, y_tr,
                        loan_va, orig_va, lm_va, om_va, y_va,
                        n_epochs=100, batch_size=256, lr=0.001,
                        weight_decay=1e-4, early_stop_patience=15, verbose_every=5)
    AttentionPairModel.plot_history(history)

    # get PanPhon sequences for realdist and balanced splits (pad with alldata maxlens)
    get_panphon_sequences(train_realdist, test_realdist)
    pad_panphon_sequences(train_realdist, test_realdist, alldata_maxlen)

    get_panphon_sequences(train_balanced, test_balanced)
    pad_panphon_sequences(train_balanced, test_balanced, alldata_maxlen)

    # get logits from the attention model for all datasets/splits
    print("Getting logits from attention model")
    train_alldata['DNN_logits'] = get_logits(model, train_alldata)
    test_alldata['DNN_logits'] = get_logits(model, test_alldata)
    train_realdist['DNN_logits'] = get_logits(model, train_realdist)
    test_realdist['DNN_logits'] = get_logits(model, test_realdist)
    train_balanced['DNN_logits'] = get_logits(model, train_balanced)
    test_balanced['DNN_logits'] = get_logits(model, test_balanced)

    # remove sequence columns from dataframes (logits column stays)
    for df in (train_alldata, test_alldata, train_realdist, test_realdist, train_balanced, test_balanced):
        df.drop(['seq_loan', 'seq_orig'], axis=1, inplace=True)

    #set the seeds for reproducibility even though we are not fine-tuning or training and the weights
    #for both these models are effectively frozen for our purpose
    torch.manual_seed(7)
    random.seed(7)
    np.random.seed(7)

    # Setting PyTorch's required configuration variables for reproducibility.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = False
    torch.use_deterministic_algorithms(False)

    PRE_TRAINED_bert_MODEL = 'jhu-clsp/mmBERT-base'
    PRE_TRAINED_xlm_MODEL = 'Alibaba-NLP/gte-multilingual-base'

    MAXTOKENS = 5
    BS = 8  # batch size

    #list of loan-original words for train sets
    l1_train_alldata = list(train_alldata["loan_word"])
    l2_train_alldata = list(train_alldata["original_word"])

    l1_train_realdist = list(train_realdist["loan_word"])
    l2_train_realdist = list(train_realdist["original_word"])

    l1_train_balanced = list(train_balanced["loan_word"])
    l2_train_balanced = list(train_balanced["original_word"])

    #list of loan-original words for test sets
    l1_test_alldata = list(test_alldata["loan_word"])
    l2_test_alldata = list(test_alldata["original_word"])

    l1_test_realdist = list(test_realdist["loan_word"])
    l2_test_realdist = list(test_realdist["original_word"])

    l1_test_balanced = list(test_balanced["loan_word"])
    l2_test_balanced = list(test_balanced["original_word"])

    print("Getting MBERT similarities")
    train_alldata['MBERT_cos_sim'] = get_mbert_cos_sims(l1_train_alldata,l2_train_alldata)
    test_alldata['MBERT_cos_sim'] = get_mbert_cos_sims(l1_test_alldata,l2_test_alldata)

    train_realdist['MBERT_cos_sim'] = train_realdist.merge(pd.concat([train_alldata,test_alldata]),\
                                                           on=['loan_word','original_word'], how="left")['MBERT_cos_sim']
    train_balanced['MBERT_cos_sim'] = train_balanced.merge(pd.concat([train_alldata,test_alldata]),\
                                                           on=['loan_word','original_word'], how="left")['MBERT_cos_sim']

    test_realdist['MBERT_cos_sim'] = test_realdist.merge(pd.concat([train_alldata,test_alldata]),\
                                                         on=['loan_word','original_word'], how="left")['MBERT_cos_sim']
    test_balanced['MBERT_cos_sim'] = test_balanced.merge(pd.concat([train_alldata,test_alldata]),\
                                                         on=['loan_word','original_word'], how="left")['MBERT_cos_sim']

    print()
    print("Getting XLM similarities")
    train_alldata['XLM_cos_sim'] = get_xlm_cos_sims(l1_train_alldata,l2_train_alldata)
    test_alldata['XLM_cos_sim'] = get_xlm_cos_sims(l1_test_alldata,l2_test_alldata)

    train_realdist['XLM_cos_sim'] = train_realdist.merge(pd.concat([train_alldata,test_alldata]),\
                                                           on=['loan_word','original_word'], how="left")['XLM_cos_sim']
    train_balanced['XLM_cos_sim'] = train_balanced.merge(pd.concat([train_alldata,test_alldata]),\
                                                           on=['loan_word','original_word'], how="left")['XLM_cos_sim']

    test_realdist['XLM_cos_sim'] = test_realdist.merge(pd.concat([train_alldata,test_alldata]),\
                                                         on=['loan_word','original_word'], how="left")['XLM_cos_sim']
    test_balanced['XLM_cos_sim'] = test_balanced.merge(pd.concat([train_alldata,test_alldata]),\
                                                         on=['loan_word','original_word'], how="left")['XLM_cos_sim']

    train_alldata.to_csv(f'{prefix}/alldata/{L1}-{L2}-train_production_alldata.csv')
    test_alldata.to_csv(f'{prefix}/alldata/{L1}-{L2}-test_production_alldata.csv')

    train_realdist.to_csv(f'{prefix}/realdist/{L1}-{L2}-train_production_realdist.csv')
    test_realdist.to_csv(f'{prefix}/realdist/{L1}-{L2}-test_production_realdist.csv')

    train_balanced.to_csv(f'{prefix}/balanced/{L1}-{L2}-train_production_balanced.csv')
    test_balanced.to_csv(f'{prefix}/balanced/{L1}-{L2}-test_production_balanced.csv')